# Falcon 9 launch records: completed web-scraping lab

All three tasks from IBM's `jupyter-labs-webscraping.ipynb` are implemented below.
The input is the actual Wikipedia revision required by the lab (9 June 2021),
not a reconstructed HTML fixture or another learner's results.
The exported 121-record Wikipedia cohort is kept separate from the 90-record
API/modeling, 101-record SQL, and 56-record dashboard cohorts in the presentation.
No final graded attempt is submitted by this notebook.

In [1]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
import json, re, unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
cache = Path('wikipedia-2021-06-09.html')
headers = {'User-Agent': 'Mozilla/5.0'}

## Task 1 — request and parse the required page
Fetch the historical revision with `requests.get`, check the HTTP response,
and parse its HTML with BeautifulSoup. An explicit, previously downloaded copy
of this exact revision is a reproducibility fallback if the server later fails.

In [2]:
used_cache = False
try:
    response = requests.get(static_url, headers=headers, timeout=20)
    response.raise_for_status()
    html_bytes = response.content
    assert b'wikitable' in html_bytes
    cache.write_bytes(html_bytes)
    http_status = response.status_code
except (requests.RequestException, AssertionError) as exc:
    if not cache.exists():
        raise
    used_cache = True
    http_status = None
    html_bytes = cache.read_bytes()
    print('Source request failed; using saved exact-revision HTML:', str(exc))

soup = BeautifulSoup(html_bytes, 'html.parser')
provenance = {'source_url': static_url,
              'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
              'http_status': http_status, 'used_cached_revision': used_cache,
              'html_bytes': len(html_bytes), 'html_sha256': sha256(html_bytes).hexdigest()}
print(json.dumps(provenance, indent=2))
print('Page title:', soup.title.get_text())
assert soup.title.get_text() == 'List of Falcon 9 and Falcon Heavy launches - Wikipedia' 

{
  "source_url": "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922",
  "retrieved_at_utc": "2026-10-23T06:22:35.855789+00:00",
  "http_status": 200,
  "used_cached_revision": false,
  "html_bytes": 3293319,
  "html_sha256": "bcaf27b72595f6374e99e18d6dae5ad24d7637e6d7acaa4fdcc4c8040067dc5e"
}
Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


## Task 2 — extract table headings
Select the launch tables by their HTML classes rather than assuming that an
unrelated navigation table will always occupy a particular position. Remove
reference markers from a copy of each cell; preserve the actual column text.

In [3]:
def clean_cell(cell):
    clone = BeautifulSoup(str(cell), 'html.parser')
    for reference in clone.select('sup.reference'):
        reference.decompose()
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', clone.get_text(' ', strip=True))).strip()

html_tables = soup.find_all('table')
launch_tables = soup.select('table.wikitable.plainrowheaders.collapsible')
assert launch_tables
column_names = [clean_cell(th) for th in launch_tables[0].find('tr').find_all('th')]
print('All HTML tables:', len(html_tables))
print('Launch tables:', len(launch_tables))
print('Extracted column names:', column_names)
assert len(column_names) == 10
assert column_names[0] == 'Flight No.' 

All HTML tables: 29
Launch tables: 9
Extracted column names: ['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## Task 3 — collect all launch records and export CSV
Only numeric flight headers identify launch records. Descriptive rows and
Falcon Heavy rows without a Falcon 9 flight number are excluded. Parse each
launch's nine data cells, split UTC date/time, and retain raw payload-mass text.
Missing or classified masses remain missing, rather than being invented as zero.

In [4]:
records = []
for table in launch_tables:
    for row in table.find_all('tr'):
        header = row.find('th', recursive=False)
        values = row.find_all('td', recursive=False)
        if header is None or not clean_cell(header).isdigit():
            continue
        if len(values) != 9:
            raise ValueError(f'Unexpected launch row width: {len(values)}')
        values = [clean_cell(cell) for cell in values]
        dt = re.search(r'(\d{1,2} [A-Za-z]+ \d{4}),?\s+(\d{1,2}:\d{2}(?::\d{2})?)', values[0])
        if dt is None:
            raise ValueError(f'Unparsed date/time: {values[0]}')
        records.append({
            'Flight No.': int(clean_cell(header)),
            'Date': dt.group(1), 'Time': dt.group(2).zfill(5 if dt.group(2).count(':') == 1 else 8),
            'Version Booster': values[1], 'Launch site': values[2],
            'Payload': values[3], 'Payload mass': values[4],
            'Orbit': values[5], 'Customer': values[6],
            'Launch outcome': values[7], 'Booster landing': values[8]})

launch_dict = {name: [row[name] for row in records] for name in records[0]}
df = pd.DataFrame(launch_dict)
df['Date'] = pd.to_datetime(df['Date'], format='%d %B %Y').dt.strftime('%Y-%m-%d')
df['Payload mass kg'] = pd.to_numeric(
    df['Payload mass'].str.extract(r'([\d,]+(?:\.\d+)?)\s*kg', expand=False)
                      .str.replace(',', '', regex=False), errors='coerce')
df.head()

,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Payload mass kg
0,1,2010-06-04,18:45,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,,LEO,SpaceX,Success,Failure (parachute),NaN
1,2,2010-12-08,15:43,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute),NaN
2,3,2012-05-22,07:44,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt,525.0
3,4,2012-10-08,00:35,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt,4700.0
4,5,2013-03-01,15:10,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt,4877.0


In [5]:
df.tail()

,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing,Payload mass kg
116,117,2021-05-09,06:42,F9 B5 ♺ B1051.10,"CCSFS , SLC-40",Starlink 27 v1.0 (60 satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship),15600.0
117,118,2021-05-15,22:56,F9 B5 ♺ B1058.8,"KSC , LC-39A",Starlink 26 v1.0 (52 Satellites) Capella -6 & ...,"~ 14,000 kg (31,000 lb)",LEO,SpaceX Capella Space and Tyvak,Success,Success (drone ship),14000.0
118,119,2021-05-26,18:59,F9 B5 ♺ B1063.2,"CCSFS , SLC-40",Starlink 28 v1.0 (60 Satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship),15600.0
119,120,2021-06-03,17:29,F9 B5 B1067.1,"KSC , LC-39A",SpaceX CRS-22 ( Dragon C209.1 ),"3,328 kg (7,337 lb)",LEO ( ISS ),NASA ( CRS ),Success,Success (drone ship),3328.0
120,121,2021-06-06,04:26,F9 B5 ♺ B1061.3,"CCSFS , SLC-40",SXM-8,"7,000 kg (15,000 lb)",GTO,Sirius XM,Success,Success (drone ship),7000.0


## Validation and results
Check independent structural constraints: 121 distinct flight numbers in order,
nine yearly launch tables, valid dates not beyond the snapshot, and CSV roundtrip.
Inspect all outcome categories and report missing masses explicitly.

In [6]:
assert len(launch_tables) == 9
assert len(df) == 121
assert df['Flight No.'].tolist() == list(range(1, 122))
assert df['Flight No.'].is_unique
assert pd.to_datetime(df['Date']).max() <= pd.Timestamp('2021-06-09')
assert df['Time'].str.fullmatch(r'\d{2}:\d{2}(?::\d{2})?').all()
assert df['Launch site'].str.len().gt(0).all()
assert df['Launch outcome'].str.len().gt(0).all()
output = Path('spacex_web_scraped.csv')
df.to_csv(output, index=False)
roundtrip = pd.read_csv(output, keep_default_na=False, na_values={'Payload mass kg': ['']})
pd.testing.assert_frame_equal(roundtrip, df, check_dtype=False)
assert roundtrip.shape == df.shape
assert roundtrip['Flight No.'].tolist() == df['Flight No.'].tolist()
summary = {'rows': len(df), 'columns': len(df.columns),
           'first_launch': df['Date'].min(), 'last_launch': df['Date'].max(),
           'launch_tables': len(launch_tables),
           'missing_payload_mass_kg': int(df['Payload mass kg'].isna().sum()),
           'launch_outcomes': df['Launch outcome'].value_counts().to_dict(),
           'csv_sha256': sha256(output.read_bytes()).hexdigest(),
           'csv_roundtrip_verified': True}
print(json.dumps(summary, indent=2))
print('Booster landing categories:')
print(df['Booster landing'].value_counts().to_string())
Path('webscraping-validation.json').write_text(
    json.dumps({'provenance': provenance, 'validation': summary}, indent=2), encoding='utf-8')

{
  "rows": 121,
  "columns": 12,
  "first_launch": "2010-06-04",
  "last_launch": "2021-06-06",
  "launch_tables": 9,
  "missing_payload_mass_kg": 5,
  "launch_outcomes": {
    "Success": 120,
    "Failure (in flight)": 1
  },
  "csv_sha256": "4e862b8a2f49ad4a7c04dae963c219217cff046c9eb6bf0abb7f56a8b8866daa",
  "csv_roundtrip_verified": true
}
Booster landing categories:
Booster landing
Success (drone ship)      64
No attempt                22
Success (ground pad)      16
Failure (drone ship)       8
Controlled (ocean)         5
Failure (parachute)        2
Uncontrolled (ocean)       2
Precluded (drone ship)     1
Failure (ground pad)       1


771

### Interpretation and limitations
This completes the required HTML request, header extraction, record parsing,
DataFrame creation, and CSV export on the official historical revision.
Wikipedia annotations are removed, but descriptive launch outcomes remain intact.
The extra numeric mass column is a transparent convenience; missing/classified
values are not imputed here. This laboratory's larger date range must not be
silently substituted into the presentation's separately documented analyses.

Lab specification: IBM Skills Network, *Web scraping Falcon 9 and Falcon Heavy
Launches Records from Wikipedia*. Source revision is linked in the code.